# Apply a Cellpose model

**Purpose.** Apply a pretrained or custom Cellpose model to a microscopy image collection.

**Recommended use.** Use for direct segmentation outside the complete mask-generation pipeline or for evaluating a newly trained model on unlabeled data.

**Primary outputs.** A label image for each processed input image and associated object measurements when enabled.

---

> Paths in this notebook are placeholders. Set `src` and any other required path to the experimental data before execution.
> spaCR writes outputs within, or immediately adjacent to, the configured source directory unless an explicit output path is set.

## 1. Verify the environment

The following cell reports the installed spaCR version. Import errors must be resolved before the analysis cells are executed. GPU acceleration is optional and depends on the selected workflow and installed computational backend.

In [ ]:
import spacr
from spacr.version import version_str

print(version_str)

## 2. API entry point

This workflow calls the following public function:

- [`spacr.spacr_cellpose.identify_masks_finetune`](https://einarolafsson.github.io/spacr/api/spacr/spacr_cellpose/index.html#spacr.spacr_cellpose.identify_masks_finetune)

```python
identify_masks_finetune(settings)
```

Parameter definitions and defaults are generated from the same public API and are listed in the settings reference below.

In [ ]:
from spacr.spacr_cellpose import identify_masks_finetune

## 3. Settings and API reference

Review the parameter definitions here, then edit the values in the categorized code cells below. Required settings must be supplied. Optional settings retain the displayed default when unchanged. Conditionally required settings are necessary only for the indicated analysis branch. Defaults and descriptions are generated from the installed spaCR version so that the notebook remains aligned with the public API.

### [`spacr.spacr_cellpose.identify_masks_finetune`](https://einarolafsson.github.io/spacr/api/spacr/spacr_cellpose/index.html#spacr.spacr_cellpose.identify_masks_finetune)


#### Input & Channels

- **`src`** *(required)* — (str, path) - Folder the current step reads from and writes into: raw images for mask generation, the merged/ folder of .npy stacks for measure, the plate root for dataset/regression steps, or the folder of .fastq.gz reads for sequencing. Outputs (stack/, masks/, measurements/measurements.db, datasets/, results/) are created inside it. A list of paths, or a "['a','b']" string, processes several plates in one run. No usable default: the settings factories fill a placeholder ('path' or '/path/to/src'), so this must be supplied.
- **`channels`** *(optional)* — (list of int) - Zero-indexed image channels kept in merged/*.npy and measured by measure_crop; each entry produces its own &lt;object&gt;_channel_&lt;n&gt;_* intensity columns. The list length fixes where masks land, so cell/nucleus/pathogen_mask_dim must shift if you change it. Preprocessing silently resets it to range(n) when it does not match the number of channel folders found. Default [0,1,2,3].
- **`grayscale`** *(optional)* — (bool) - Force the Cellpose channel pair to [0, 0] so the network treats the input as a single combined channel, overriding the [cytoplasm, nucleus] pair otherwise inferred from model_name (cyto -&gt; [1,0], cyto2 -&gt; [2,1], nucleus -&gt; [0,0]). Leave it on for single-channel inputs; switch it off only when feeding a genuine two-channel stack. Default True.
- **`invert`** *(optional)* — (bool) - Invert intensities as each image is loaded, pixel -&gt; dtype_max - pixel (255 - x for uint8). Switch it on for brightfield or phase-contrast data where objects are darker than the background, since Cellpose expects bright objects on a dark field; leave it off for fluorescence. Default False.
- **`normalize`** *(optional)* — (bool) - Percentile-normalize each image channel (2nd to 98th percentile, clipped to 0-1) before display or model input; in the activation-map tool this rescales the image the CAM/saliency heatmap is drawn over. Turn it on when raw channels are too dim to read under the overlay. Affects display and input scaling only, never stored pixels. Default True.
- **`percentiles`** *(optional)* — (list) - Two percentiles [low, high] used to rescale each channel of each image to 0-1 before segmentation, e.g. [2, 98]. Narrowing the window boosts contrast on dim objects but clips bright ones. Set None to derive them automatically: low fixed at 2, high the first of 98/99/99.9/99.99/99.999 exceeding background * Signal_to_noise. Default None in the Cellpose steps.

#### Model

- **`model_name`** *(conditionally required)* — (str) - Cellpose model to segment with. Cellpose 4 ships exactly one, 'cpsam'; the pre-SAM names ('cyto', 'cyto2', 'cyto3', 'nuclei') are accepted so old settings files load, but they are mapped to 'cpsam' and reported, because Cellpose resolves them to cpsam silently anyway. Of the three parameters that used to distinguish models, only diameter still does anything under Cellpose 4 (eval rescales the image by 30/diameter); model_type and diam_mean are logged as 'not used in v4.0.1+' and dropped. Leave at 'cpsam' unless you are loading a custom CPSAM checkpoint. Default 'cpsam'.
- **`custom_model`** *(conditionally required)* — (str) - Path to a saved Cellpose model, loaded as pretrained_model by the mask-finetune tool. When set, model_type is passed as None and diameter as diam_mean (which Cellpose 4.x ignores with a warning), but model_name is STILL read: it selects the channel pair sent to model.eval. So a custom model with the wrong model_name segments the wrong channels. Default None.
- **`diameter`** *(optional)* — (float) - DEPRECATED. Expected object diameter in pixels, passed to model.eval(diameter=...) by the mask-finetune tool and check_cellpose_models. Cellpose resizes each image by 30/diameter so objects match the network's ~30 px working size, so a value BELOW the true size upscales the image and one above downscales it. Prefer the per-object diameter settings; this one remains only for those two tools. Default 30.

#### Detection Thresholds

- **`CP_prob`** *(optional)* — (float) - Cellpose cellprob_threshold: the cell-probability cut-off applied to the network output when deciding which pixels belong to an object. Lower it (typically toward -6) to recover dim or partly detected objects and grow existing masks; raise it (toward 6) to drop faint false positives and shrink masks. Default 0.
- **`flow_threshold`** *(optional)* — (float) - Cellpose flow_threshold: the maximum allowed error between the predicted flow field and the flows recomputed from each candidate mask; masks above it are discarded. Raise it to keep more objects, including irregularly shaped ones; lower it to reject poorly formed masks and reduce false positives. Default 0.4.
- **`rescale`** *(optional)* — (bool) - Let Cellpose rescale each image by 30/diameter before segmenting, so objects arrive at the size the model expects. Turn off only when the diameter is already correct for the model. Default False.
- **`resample`** *(optional)* — (bool) - Passed to Cellpose model.eval: run the mask-tracking dynamics at full image resolution instead of on the downsampled network grid. Enabling it gives smoother, better-fitting object outlines at the cost of time and memory, and helps most when objects differ a lot from the model's training diameter. Default False; the object pipeline sets True for cell/nucleus and False for pathogen.
- **`fill_in`** *(optional)* — (bool) - Post-process each Cellpose mask with fill_holes_in_mask in the mask-finetune and plaque tools: the mask is re-labelled by connectivity over all non-zero pixels, then interior holes are filled component by component. Because re-labelling IGNORES the original label values, two touching objects become one -- so this repairs hollow objects at the risk of merging adjacent ones. Default False.

#### Image Geometry

- **`resize`** *(optional)* — (bool or float) - Resize every image to target_height x target_width before running Cellpose, then scale the returned mask back to the original dimensions with nearest-neighbour interpolation so measurements stay in original pixels. Turn it on to bring oversized fields to the scale a model was trained at, or to cut GPU memory. Requires target_height and target_width. Default False (True for plaque analysis).
- **`target_height`** *(optional)* — (int) - Height in pixels that images are resized to before segmentation; masks are scaled back to the original dimensions afterwards. Only applied when both target_height and target_width are set (and, on the non-normalized path, when resize is True). Use it to match the field size the model was trained at. Default None, which disables resizing; 1120 for plaque analysis.
- **`target_width`** *(optional)* — (int) - Width in pixels that images are resized to before segmentation; masks are scaled back to the original dimensions afterwards. Only applied when both target_width and target_height are set (and, on the non-normalized path, when resize is True). Use it to match the field size the model was trained at. Default None, which disables resizing; 1120 for plaque analysis.

#### Background & Denoising

- **`remove_background`** *(optional)* — (bool) - Hard-clip every pixel below the 'background' value to zero before normalization and segmentation. Use it when a channel carries a bright, even haze that inflates the normalization floor; leave it off for dim or already flat-fielded data, since the clip silently deletes faint real signal. Default False.
- **`background`** *(optional)* — (float) - Per-channel background level in raw intensity units. Pixels below it are zeroed when remove_background is on, and it is multiplied by Signal_to_noise to set the upper anchor for normalization. Raise it if faint haze survives; set it too high and dim real objects vanish. Default 100 (200 for Cellpose training and plaque analysis).
- **`Signal_to_noise`** *(optional)* — (int) - Multiplier on background setting the signal threshold (background * Signal_to_noise) used when normalising for Cellpose. Per channel, spaCR takes the first of the 98th, 99th, 99.9th, 99.99th and 99.999th percentiles that exceeds the threshold and rescales to it. RAISE for less clipping and dimmer output; LOWER to stretch faint objects harder at the cost of saturating bright ones. If no percentile clears the threshold the range collapses to the 2nd percentile, which is the sign the value is far too high. Ignored when percentiles is set. Default 10 (5 in check_cellpose_models).

#### Output & Runtime

- **`save`** *(optional)* — (bool or list of bool) - Whether to save masks to disk. Can be a list of three booleans for [cell, nucleus, pathogen] independently. Default varies by module -- False for most, True for the sequencing and regression paths.
- **`batch_size`** *(optional)* — (int) - How many images are held and processed together in one pass: field stacks during normalization and Cellpose segmentation, crops per step during classifier training and activation maps. Raising it speeds runs up but increases RAM/VRAM roughly linearly; lower it on out-of-memory errors. Defaults: 50 for mask generation, 64 for training.
- **`verbose`** *(optional)* — (bool) - Print extra run detail instead of the minimal log: the resolved settings table, the channel and model choices per object type, per-table row counts, and how many objects survive each filter. It only adds console output, so turn it on when object counts come out unexpected and you need to see which stage removed them. The default differs per pipeline -- True for mask, UMAP, screen analysis, barcode mapping and Cellpose training; False for measure, the plotting helpers and regression.

## 4. Run

After editing the settings cell, run the function cell immediately below it. Long operations report progress through spaCR's logging system; set `SPACR_LOG_LEVEL=DEBUG` before starting Jupyter for more detail.

In [ ]:
settings = {
    # Input & Channels
    # Required settings
    'src': 'path',
    # Optional settings
    'channels': [0, 0],
    'grayscale': True,
    'invert': False,
    'normalize': True,
    'percentiles': None,

    # Model
    # Conditionally required settings
    'model_name': 'cpsam',
    'custom_model': None,
    # Optional settings
    'diameter': 30,

    # Detection Thresholds
    # Optional settings
    'CP_prob': 0,
    'flow_threshold': 0.4,
    'rescale': False,
    'resample': False,
    'fill_in': True,

    # Image Geometry
    # Optional settings
    'resize': False,
    'target_height': None,
    'target_width': None,

    # Background & Denoising
    # Optional settings
    'remove_background': False,
    'background': 100,
    'Signal_to_noise': 10,

    # Output & Runtime
    # Optional settings
    'save': False,
    'batch_size': 50,
    'verbose': False,
}

In [ ]:
identify_masks_finetune(settings)

## Outputs and next steps

A label image for each processed input image and associated object measurements when enabled.

Output directories remain associated with the source dataset, which preserves plate-level provenance across subsequent spaCR workflows.

### Related documentation

- [Graphical workflow tutorials](https://einarolafsson.github.io/spacr/tutorials/)
- [Python API reference](https://einarolafsson.github.io/spacr/python_api.html)